In [1]:
!pip install -q tensorflow numpy scikit-learn tqdm opencv-python-headless

import tensorflow as tf

print("TensorFlow:", tf.__version__)
print("GPUs:", tf.config.list_physical_devices("GPU"))
!nvidia-smi || true

TensorFlow: 2.20.0
GPUs: [PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]
Fri May  8 07:23:15 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   42C    P8              9W /   70W |       3MiB /  15360MiB |      0%      Default |
|                                         |     

In [2]:
from pathlib import Path

SAMPLES_PER_CLASS = 10000
MAX_LEN = 256
IMAGE_SIZE = 64
BATCH_SIZE = 128
EPOCHS = 40
SEED = 42

DATA_DIR = Path("/content/data")
NDJSON_DIR = DATA_DIR / "quickdraw_simplified"
PROCESSED_DIR = DATA_DIR / "processed_fused"
MODEL_DIR = Path("/content/quickdraw_fused_tflite")

NDJSON_DIR.mkdir(parents=True, exist_ok=True)
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

CLASSES = [
    "The Mona Lisa",
    "apple",
    "banana",
    "cat",
    "dog",
    "fish",
    "bird",
    "airplane",
    "car",
    "bicycle",
    "bus",
    "tree",
    "flower",
    "house",
    "chair",
    "table",
    "cup",
    "fork",
    "umbrella",
    "star",
    "moon",
    "sun",
    "cloud",
    "crown",
    "pizza",
    "ice cream",
    "book",
    "clock",
    "eye",
    "face",
]

print("Classes:", len(CLASSES))

Classes: 30


In [3]:
import subprocess

for class_name in CLASSES:
    src = f"gs://quickdraw_dataset/full/simplified/{class_name}.ndjson"
    dst = NDJSON_DIR / f"{class_name}.ndjson"

    if dst.exists():
        print(f"Already exists: {dst}")
        continue

    print(f"Downloading: {class_name}")
    subprocess.run(["gsutil", "cp", src, str(dst)], check=True)

print("Download complete.")

Downloading: The Mona Lisa
Downloading: apple
Downloading: banana
Downloading: cat
Downloading: dog
Downloading: fish
Downloading: bird
Downloading: airplane
Downloading: car
Downloading: bicycle
Downloading: bus
Downloading: tree
Downloading: flower
Downloading: house
Downloading: chair
Downloading: table
Downloading: cup
Downloading: fork
Downloading: umbrella
Downloading: star
Downloading: moon
Downloading: sun
Downloading: cloud
Downloading: crown
Downloading: pizza
Downloading: ice cream
Downloading: book
Downloading: clock
Downloading: eye
Downloading: face
Download complete.


In [4]:
import json
import random
import cv2
import numpy as np
from tqdm import tqdm
from sklearn.model_selection import train_test_split

random.seed(SEED)
np.random.seed(SEED)


def normalize_strokes(drawing):
    """
    QuickDraw simplified format:
    [
      [[x0, x1, ...], [y0, y1, ...]],
      [[x0, x1, ...], [y0, y1, ...]]
    ]

    Returns list of strokes:
    [
      [(x, y), (x, y), ...],
      ...
    ]

    Coordinates are normalized into 0..255.
    """
    points = []

    for stroke in drawing:
        if len(stroke) != 2:
            continue

        xs, ys = stroke

        for x, y in zip(xs, ys):
            points.append((float(x), float(y)))

    if not points:
        return []

    xs_all = [p[0] for p in points]
    ys_all = [p[1] for p in points]

    min_x, max_x = min(xs_all), max(xs_all)
    min_y, max_y = min(ys_all), max(ys_all)

    width = max(max_x - min_x, 1.0)
    height = max(max_y - min_y, 1.0)
    scale = 255.0 / max(width, height)

    normalized = []

    for stroke in drawing:
        if len(stroke) != 2:
            continue

        xs, ys = stroke
        new_stroke = []

        for x, y in zip(xs, ys):
            nx = (float(x) - min_x) * scale
            ny = (float(y) - min_y) * scale
            new_stroke.append((nx, ny))

        if len(new_stroke) >= 2:
            normalized.append(new_stroke)

    return normalized


def strokes_to_sequence(drawing, max_len=MAX_LEN):
    """
    Output: [max_len, 5]
    Features: [dx, dy, pen_down, pen_up, end]

    dx/dy are normalized by /255.0.
    """
    strokes = normalize_strokes(drawing)

    seq = []
    prev_x = None
    prev_y = None

    for stroke in strokes:
        for i, (x, y) in enumerate(stroke):
            if prev_x is None:
                dx = 0.0
                dy = 0.0
            else:
                dx = x - prev_x
                dy = y - prev_y

            is_last_point = i == len(stroke) - 1

            pen_down = 0.0 if is_last_point else 1.0
            pen_up = 1.0 if is_last_point else 0.0
            end = 0.0

            seq.append([
                dx / 255.0,
                dy / 255.0,
                pen_down,
                pen_up,
                end,
            ])

            prev_x = x
            prev_y = y

            if len(seq) >= max_len - 1:
                break

        if len(seq) >= max_len - 1:
            break

    seq.append([0.0, 0.0, 0.0, 0.0, 1.0])

    while len(seq) < max_len:
        seq.append([0.0, 0.0, 0.0, 0.0, 0.0])

    return np.array(seq[:max_len], dtype=np.float32)


def render_drawing_to_image(drawing, image_size=IMAGE_SIZE):
    """
    Output: [image_size, image_size, 1] uint8.

    Convention:
    white background = 255
    black stroke = 0

    Later, the tf.data pipeline casts to float32 and divides by 255.0:
    white = 1.0
    black = 0.0
    """
    strokes = normalize_strokes(drawing)

    img = np.full((256, 256), 255, dtype=np.uint8)

    for stroke in strokes:
        if len(stroke) < 2:
            continue

        pts = np.array(stroke, dtype=np.float32)
        pts = np.round(pts).astype(np.int32)
        pts = np.clip(pts, 0, 255)

        for i in range(1, len(pts)):
            p1 = tuple(pts[i - 1])
            p2 = tuple(pts[i])
            cv2.line(img, p1, p2, color=0, thickness=4, lineType=cv2.LINE_AA)

    img = cv2.resize(img, (image_size, image_size), interpolation=cv2.INTER_AREA)
    img = img[:, :, None]

    return img.astype(np.uint8)

In [5]:
TOTAL_SAMPLES = SAMPLES_PER_CLASS * len(CLASSES)

IMAGE_MEMMAP_PATH = PROCESSED_DIR / "x_image_uint8.dat"
STROKE_MEMMAP_PATH = PROCESSED_DIR / "x_stroke_float32.dat"
Y_PATH = PROCESSED_DIR / "y.npy"

x_image_mm = np.memmap(
    IMAGE_MEMMAP_PATH,
    dtype=np.uint8,
    mode="w+",
    shape=(TOTAL_SAMPLES, IMAGE_SIZE, IMAGE_SIZE, 1),
)

x_stroke_mm = np.memmap(
    STROKE_MEMMAP_PATH,
    dtype=np.float32,
    mode="w+",
    shape=(TOTAL_SAMPLES, MAX_LEN, 5),
)

y = np.zeros((TOTAL_SAMPLES,), dtype=np.int64)

write_idx = 0

for label_id, class_name in enumerate(CLASSES):
    path = NDJSON_DIR / f"{class_name}.ndjson"
    count = 0

    with open(path, "r", encoding="utf-8") as f:
        for line in tqdm(f, desc=f"Preprocessing {class_name}"):
            row = json.loads(line)

            if not row.get("recognized", False):
                continue

            drawing = row.get("drawing")
            if not drawing:
                continue

            image_tensor = render_drawing_to_image(drawing)
            stroke_tensor = strokes_to_sequence(drawing)

            x_image_mm[write_idx] = image_tensor
            x_stroke_mm[write_idx] = stroke_tensor
            y[write_idx] = label_id

            write_idx += 1
            count += 1

            if count >= SAMPLES_PER_CLASS:
                break

print("Requested total:", TOTAL_SAMPLES)
print("Actual written:", write_idx)

# Trim metadata to actual written count.
np.save(Y_PATH, y[:write_idx])

x_image_mm.flush()
x_stroke_mm.flush()

with open(PROCESSED_DIR / "labels.json", "w", encoding="utf-8") as f:
    json.dump(CLASSES, f, indent=2)

with open(PROCESSED_DIR / "dataset_info.json", "w", encoding="utf-8") as f:
    json.dump({
        "actual_samples": int(write_idx),
        "samples_per_class": SAMPLES_PER_CLASS,
        "classes": CLASSES,
        "image_shape": [IMAGE_SIZE, IMAGE_SIZE, 1],
        "stroke_shape": [MAX_LEN, 5],
        "image_convention": "white=1.0 black_stroke=0.0 after /255.0",
    }, f, indent=2)

print("Preprocessing complete.")

Preprocessing The Mona Lisa: 10835it [00:23, 469.09it/s]
Preprocessing apple: 10376it [00:07, 1468.97it/s]
Preprocessing banana: 10755it [00:05, 2080.92it/s]
Preprocessing cat: 11951it [00:09, 1286.32it/s]
Preprocessing dog: 10600it [00:09, 1132.31it/s]
Preprocessing fish: 10606it [00:05, 1913.18it/s]
Preprocessing bird: 11893it [00:07, 1503.88it/s]
Preprocessing airplane: 11151it [00:06, 1842.98it/s]
Preprocessing car: 11374it [00:08, 1357.19it/s]
Preprocessing bicycle: 10309it [00:08, 1148.08it/s]
Preprocessing bus: 11547it [00:08, 1388.34it/s]
Preprocessing tree: 10473it [00:08, 1229.99it/s]
Preprocessing flower: 10449it [00:11, 944.24it/s] 
Preprocessing house: 10214it [00:05, 1897.24it/s]
Preprocessing chair: 10951it [00:06, 1824.71it/s]
Preprocessing table: 10470it [00:05, 2074.29it/s]
Preprocessing cup: 10763it [00:07, 1535.39it/s]
Preprocessing fork: 10591it [00:04, 2159.82it/s]
Preprocessing umbrella: 10308it [00:06, 1487.12it/s]
Preprocessing star: 10376it [00:05, 2034.20it/s

Requested total: 300000
Actual written: 300000
Preprocessing complete.


In [6]:
with open(PROCESSED_DIR / "dataset_info.json", "r", encoding="utf-8") as f:
    dataset_info = json.load(f)

N = dataset_info["actual_samples"]
y = np.load(Y_PATH)

indices = np.arange(N)

train_idx, temp_idx, y_train_idx, y_temp_idx = train_test_split(
    indices,
    y,
    test_size=0.2,
    random_state=SEED,
    stratify=y,
)

val_idx, test_idx, y_val_idx, y_test_idx = train_test_split(
    temp_idx,
    y_temp_idx,
    test_size=0.5,
    random_state=SEED,
    stratify=y_temp_idx,
)

np.save(PROCESSED_DIR / "train_idx.npy", train_idx)
np.save(PROCESSED_DIR / "val_idx.npy", val_idx)
np.save(PROCESSED_DIR / "test_idx.npy", test_idx)

print("Train:", len(train_idx))
print("Val:", len(val_idx))
print("Test:", len(test_idx))

Train: 240000
Val: 30000
Test: 30000


In [7]:
import math
import tensorflow as tf

x_image_read = np.memmap(
    IMAGE_MEMMAP_PATH,
    dtype=np.uint8,
    mode="r",
    shape=(N, IMAGE_SIZE, IMAGE_SIZE, 1),
)

x_stroke_read = np.memmap(
    STROKE_MEMMAP_PATH,
    dtype=np.float32,
    mode="r",
    shape=(N, MAX_LEN, 5),
)

y_read = np.load(Y_PATH)


class FusedSequence(tf.keras.utils.Sequence):
    def __init__(self, indices, batch_size=BATCH_SIZE, shuffle=True):
        self.indices = np.array(indices)
        self.batch_size = batch_size
        self.shuffle = shuffle
        self.on_epoch_end()

    def __len__(self):
        return math.ceil(len(self.indices) / self.batch_size)

    def on_epoch_end(self):
        if self.shuffle:
            np.random.shuffle(self.indices)

    def __getitem__(self, batch_num):
        batch_indices = self.indices[
            batch_num * self.batch_size : (batch_num + 1) * self.batch_size
        ]

        image_batch = x_image_read[batch_indices].astype(np.float32) / 255.0
        stroke_batch = x_stroke_read[batch_indices].astype(np.float32)
        y_batch = y_read[batch_indices].astype(np.int64)

        return {
            "image_input": image_batch,
            "stroke_input": stroke_batch,
        }, y_batch


train_seq = FusedSequence(train_idx, batch_size=BATCH_SIZE, shuffle=True)
val_seq = FusedSequence(val_idx, batch_size=BATCH_SIZE, shuffle=False)
test_seq = FusedSequence(test_idx, batch_size=BATCH_SIZE, shuffle=False)

print("Train batches:", len(train_seq))
print("Val batches:", len(val_seq))
print("Test batches:", len(test_seq))

Train batches: 1875
Val batches: 235
Test batches: 235


In [8]:
from tensorflow import keras
from tensorflow.keras import layers

num_classes = len(CLASSES)

# Image branch
image_input = keras.Input(
    shape=(IMAGE_SIZE, IMAGE_SIZE, 1),
    name="image_input",
)

x_img = layers.Conv2D(32, 3, padding="same", activation="relu")(image_input)
x_img = layers.BatchNormalization()(x_img)
x_img = layers.MaxPooling2D()(x_img)

x_img = layers.Conv2D(64, 3, padding="same", activation="relu")(x_img)
x_img = layers.BatchNormalization()(x_img)
x_img = layers.MaxPooling2D()(x_img)

x_img = layers.Conv2D(128, 3, padding="same", activation="relu")(x_img)
x_img = layers.BatchNormalization()(x_img)
x_img = layers.MaxPooling2D()(x_img)

x_img = layers.GlobalAveragePooling2D()(x_img)
x_img = layers.Dense(128, activation="relu")(x_img)


# Stroke branch
stroke_input = keras.Input(
    shape=(MAX_LEN, 5),
    name="stroke_input",
)

x_stroke = layers.Conv1D(64, kernel_size=5, padding="same", activation="relu")(stroke_input)
x_stroke = layers.BatchNormalization()(x_stroke)
x_stroke = layers.MaxPooling1D(pool_size=2)(x_stroke)

x_stroke = layers.Conv1D(128, kernel_size=5, padding="same", activation="relu")(x_stroke)
x_stroke = layers.BatchNormalization()(x_stroke)
x_stroke = layers.MaxPooling1D(pool_size=2)(x_stroke)

x_stroke = layers.Conv1D(256, kernel_size=3, padding="same", activation="relu")(x_stroke)
x_stroke = layers.BatchNormalization()(x_stroke)

x_stroke = layers.GlobalAveragePooling1D()(x_stroke)
x_stroke = layers.Dense(128, activation="relu")(x_stroke)


# Fusion
x = layers.Concatenate(name="fused_features")([x_img, x_stroke])
x = layers.Dense(256, activation="relu")(x)
x = layers.Dropout(0.30)(x)

outputs = layers.Dense(
    num_classes,
    activation="softmax",
    name="class_probs",
)(x)

model = keras.Model(
    inputs={
        "image_input": image_input,
        "stroke_input": stroke_input,
    },
    outputs=outputs,
    name="quickdraw_fused_image_stroke",
)

model.compile(
    optimizer=keras.optimizers.Adam(learning_rate=1e-3),
    loss="sparse_categorical_crossentropy",
    metrics=[
        "accuracy",
        keras.metrics.SparseTopKCategoricalAccuracy(k=3, name="top3"),
    ],
)

model.summary()

Model: "quickdraw_fused_image_stroke"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ image_input         │ (None, 64, 64, 1) │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d (Conv2D)     │ (None, 64, 64,    │        320 │ image_input[0][0] │
│                     │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ stroke_input        │ (None, 256, 5)    │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 64, 64,    │        128 │ conv2d[0][0]      │
│ (BatchNormalizatio… │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 256, 64)   │      1,664 │ stroke_input[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d       │ (None, 32, 32,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 32)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 256, 64)   │        256 │ conv1d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_1 (Conv2D)   │ (None, 32, 32,    │     18,496 │ max_pooling2d[0]… │
│                     │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d       │ (None, 128, 64)   │          0 │ batch_normalizat… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 32, 32,    │        256 │ conv2d_1[0][0]    │
│ (BatchNormalizatio… │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 128, 128)  │     41,088 │ max_pooling1d[0]… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_1     │ (None, 16, 16,    │          0 │ batch_normalizat… │
│ (MaxPooling2D)      │ 64)               │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 128, 128)  │        512 │ conv1d_1[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv2d_2 (Conv2D)   │ (None, 16, 16,    │     73,856 │ max_pooling2d_1[… │
│                     │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling1d_1     │ (None, 64, 128)   │          0 │ batch_normalizat… │
│ (MaxPooling1D)      │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 16, 16,    │        512 │ conv2d_2[0][0]    │
│ (BatchNormalizatio… │ 128)              │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_2 (Conv1D)   │ (None, 64, 256)   │     98,560 │ max_pooling1d_1[… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ max_pooling2d_2     │ (None, 8, 8, 128) │          0 │ batch_normalizat

 Total params: 359,582 (1.37 MB)

 Trainable params: 358,238 (1.37 MB)

 Non-trainable params: 1,344 (5.25 KB)

In [9]:
callbacks = [
    keras.callbacks.ModelCheckpoint(
        filepath=str(MODEL_DIR / "best.keras"),
        monitor="val_top3",
        mode="max",
        save_best_only=True,
        verbose=1,
    ),
    keras.callbacks.EarlyStopping(
        monitor="val_top3",
        mode="max",
        patience=6,
        restore_best_weights=True,
        verbose=1,
    ),
    keras.callbacks.ReduceLROnPlateau(
        monitor="val_top3",
        mode="max",
        factor=0.5,
        patience=2,
        min_lr=1e-5,
        verbose=1,
    ),
]

history = model.fit(
    train_seq,
    validation_data=val_seq,
    epochs=EPOCHS,
    callbacks=callbacks,
)

model.save(MODEL_DIR / "final.keras")

with open(MODEL_DIR / "labels.json", "w", encoding="utf-8") as f:
    json.dump(CLASSES, f, indent=2)

with open(MODEL_DIR / "history.json", "w", encoding="utf-8") as f:
    json.dump(history.history, f, indent=2)

print("Saved Keras model and labels.")

Epoch 1/40


/usr/local/lib/python3.12/dist-packages/keras/src/trainers/data_adapters/py_dataset_adapter.py:121: UserWarning: Your `PyDataset` class should call `super().__init__(**kwargs)` in its constructor. `**kwargs` can include `workers`, `use_multiprocessing`, `max_queue_size`. Do not pass these arguments to `fit()`, as they will be ignored.
  self._warn_if_super_not_called()


1875/1875 ━━━━━━━━━━━━━━━━━━━━ 0s 26ms/step - accuracy: 0.7374 - loss: 0.8936 - top3: 0.8813
Epoch 1: val_top3 improved from None to 0.77657, saving model to /content/quickdraw_fused_tflite/best.keras

Epoch 1: finished saving model to /content/quickdraw_fused_tflite/best.keras
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 74s 29ms/step - accuracy: 0.8525 - loss: 0.4908 - top3: 0.9527 - val_accuracy: 0.4943 - val_loss: 1.7052 - val_top3: 0.7766 - learning_rate: 0.0010
Epoch 2/40
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.9353 - loss: 0.2084 - top3: 0.9881
Epoch 2: val_top3 improved from 0.77657 to 0.93553, saving model to /content/quickdraw_fused_tflite/best.keras

Epoch 2: finished saving model to /content/quickdraw_fused_tflite/best.keras
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 54s 29ms/step - accuracy: 0.9384 - loss: 0.1993 - top3: 0.9889 - val_accuracy: 0.7842 - val_loss: 0.7269 - val_top3: 0.9355 - learning_rate: 0.0010
Epoch 3/40
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 0s 28ms/step - accuracy: 0.9

In [10]:
best_model = keras.models.load_model(MODEL_DIR / "best.keras")

loss, acc, top3 = best_model.evaluate(test_seq)

print("Test loss:", loss)
print("Test top-1 accuracy:", acc)
print("Test top-3 accuracy:", top3)

235/235 ━━━━━━━━━━━━━━━━━━━━ 4s 11ms/step - accuracy: 0.9665 - loss: 0.1577 - top3: 0.9952
Test loss: 0.15773621201515198
Test top-1 accuracy: 0.9664999842643738
Test top-3 accuracy: 0.9952333569526672


In [11]:
from sklearn.metrics import classification_report, confusion_matrix

all_probs = []
all_true = []

for batch_x, batch_y in test_seq:
    probs = best_model.predict(batch_x, verbose=0)
    all_probs.append(probs)
    all_true.append(batch_y)

probs = np.concatenate(all_probs, axis=0)
true_y = np.concatenate(all_true, axis=0)

preds = probs.argmax(axis=1)

print(classification_report(true_y, preds, target_names=CLASSES, digits=4))

cm = confusion_matrix(true_y, preds)

confusions = []

for true_id in range(len(CLASSES)):
    for pred_id in range(len(CLASSES)):
        if true_id == pred_id:
            continue

        count = cm[true_id, pred_id]
        if count > 0:
            confusions.append((count, CLASSES[true_id], CLASSES[pred_id]))

confusions.sort(reverse=True)

print("Worst confusions:")
for count, true_label, pred_label in confusions[:30]:
    print(f"{true_label} → {pred_label}: {count}")

               precision    recall  f1-score   support

The Mona Lisa     0.9751    0.9810    0.9781      1000
        apple     0.9929    0.9840    0.9884      1000
       banana     0.9451    0.9470    0.9461      1000
          cat     0.9141    0.9370    0.9254      1000
          dog     0.8848    0.8680    0.8763      1000
         fish     0.9799    0.9740    0.9769      1000
         bird     0.9193    0.9230    0.9212      1000
     airplane     0.9589    0.9560    0.9574      1000
          car     0.9681    0.9700    0.9690      1000
      bicycle     0.9870    0.9900    0.9885      1000
          bus     0.9759    0.9700    0.9729      1000
         tree     0.9558    0.9730    0.9643      1000
       flower     0.9679    0.9640    0.9659      1000
        house     0.9829    0.9800    0.9815      1000
        chair     0.9656    0.9820    0.9737      1000
        table     0.9757    0.9620    0.9688      1000
          cup     0.9748    0.9690    0.9719      1000
         

In [12]:
converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
tflite_model = converter.convert()

tflite_path = MODEL_DIR / "quickdraw_fused_model_float32.tflite"

with open(tflite_path, "wb") as f:
    f.write(tflite_model)

print("Saved:", tflite_path)
print("Size MB:", tflite_path.stat().st_size / 1024 / 1024)

/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: {'image_input': 'image_input', 'stroke_input': 'stroke_input'}
Received: inputs=['Tensor(shape=(None, 64, 64, 1))', 'Tensor(shape=(None, 256, 5))']
  warnings.warn(msg)


Saved artifact at '/tmp/tmpmo3ds_uy'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): List[TensorSpec(shape=(None, 64, 64, 1), dtype=tf.float32, name='image_input'), TensorSpec(shape=(None, 256, 5), dtype=tf.float32, name='stroke_input')]
Output Type:
  TensorSpec(shape=(None, 30), dtype=tf.float32, name=None)
Captures:
  135227338095696: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135227331418576: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135227315728272: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135227315725968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135227331418192: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135227331415888: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135227331416080: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135227331418000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135227315725008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1352273

In [13]:
converter = tf.lite.TFLiteConverter.from_keras_model(best_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
converter.target_spec.supported_types = [tf.float16]

tflite_float16 = converter.convert()

tflite_float16_path = MODEL_DIR / "quickdraw_fused_model_float16.tflite"

with open(tflite_float16_path, "wb") as f:
    f.write(tflite_float16)

print("Saved:", tflite_float16_path)
print("Size MB:", tflite_float16_path.stat().st_size / 1024 / 1024)

/usr/local/lib/python3.12/dist-packages/keras/src/models/functional.py:241: UserWarning: The structure of `inputs` doesn't match the expected structure.
Expected: {'image_input': 'image_input', 'stroke_input': 'stroke_input'}
Received: inputs=['Tensor(shape=(None, 64, 64, 1))', 'Tensor(shape=(None, 256, 5))']
  warnings.warn(msg)


Saved artifact at '/tmp/tmpj5ior6a_'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): List[TensorSpec(shape=(None, 64, 64, 1), dtype=tf.float32, name='image_input'), TensorSpec(shape=(None, 256, 5), dtype=tf.float32, name='stroke_input')]
Output Type:
  TensorSpec(shape=(None, 30), dtype=tf.float32, name=None)
Captures:
  135227338095696: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135227331418576: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135227315728272: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135227315725968: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135227331418192: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135227331415888: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135227331416080: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135227331418000: TensorSpec(shape=(), dtype=tf.resource, name=None)
  135227315725008: TensorSpec(shape=(), dtype=tf.resource, name=None)
  1352273

In [14]:
interpreter = tf.lite.Interpreter(model_path=str(tflite_path))
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

print("Input details:")
for d in input_details:
    print(d["name"], d["shape"], d["dtype"], "index:", d["index"])

print("\nOutput details:")
for d in output_details:
    print(d["name"], d["shape"], d["dtype"], "index:", d["index"])


# Build one sample from test_seq.
sample_x, sample_y = test_seq[0]

sample_image = sample_x["image_input"][0:1].astype(np.float32)
sample_stroke = sample_x["stroke_input"][0:1].astype(np.float32)

for d in input_details:
    name = d["name"].lower()
    shape = list(d["shape"])

    if "image" in name or shape == [1, IMAGE_SIZE, IMAGE_SIZE, 1]:
        interpreter.set_tensor(d["index"], sample_image)
        print("Set image tensor:", d["name"])
    elif "stroke" in name or shape == [1, MAX_LEN, 5]:
        interpreter.set_tensor(d["index"], sample_stroke)
        print("Set stroke tensor:", d["name"])
    else:
        raise RuntimeError(f"Unknown input tensor: name={d['name']} shape={shape}")

interpreter.invoke()

output = interpreter.get_tensor(output_details[0]["index"])[0]
top_ids = output.argsort()[-3:][::-1]

true_label = CLASSES[int(sample_y[0])]

print("\nTrue label:", true_label)
print("Top 3:")
for idx in top_ids:
    print(CLASSES[idx], float(output[idx]))

Input details:
serving_default_image_input:0 [ 1 64 64  1] <class 'numpy.float32'> index: 0
serving_default_stroke_input:0 [  1 256   5] <class 'numpy.float32'> index: 1

Output details:
StatefulPartitionedCall_1:0 [ 1 30] <class 'numpy.float32'> index: 82
Set image tensor: serving_default_image_input:0
Set stroke tensor: serving_default_stroke_input:0

True label: book
Top 3:
book 0.9996041655540466
cup 0.0003770272305700928
chair 1.884230186988134e-05


/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


In [15]:
model_info = {
    "model_name": "quickdraw_fused_image_stroke",
    "input_mode": "fused",
    "inputs": {
        "image": {
            "shape": [1, IMAGE_SIZE, IMAGE_SIZE, 1],
            "dtype": "float32",
            "convention": "white_background_1_black_stroke_0",
        },
        "stroke": {
            "shape": [1, MAX_LEN, 5],
            "dtype": "float32",
            "sequence_format": ["dx", "dy", "pen_down", "pen_up", "end"],
            "dx_dy_scaled_by": 255.0,
        },
    },
    "output_shape": [1, len(CLASSES)],
    "output_dtype": "float32",
    "classes": CLASSES,
    "class_count": len(CLASSES),
}

with open(MODEL_DIR / "model_info.json", "w", encoding="utf-8") as f:
    json.dump(model_info, f, indent=2)

print(json.dumps(model_info, indent=2))

{
  "model_name": "quickdraw_fused_image_stroke",
  "input_mode": "fused",
  "inputs": {
    "image": {
      "shape": [
        1,
        64,
        64,
        1
      ],
      "dtype": "float32",
      "convention": "white_background_1_black_stroke_0"
    },
    "stroke": {
      "shape": [
        1,
        256,
        5
      ],
      "dtype": "float32",
      "sequence_format": [
        "dx",
        "dy",
        "pen_down",
        "pen_up",
        "end"
      ],
      "dx_dy_scaled_by": 255.0
    }
  },
  "output_shape": [
    1,
    30
  ],
  "output_dtype": "float32",
  "classes": [
    "The Mona Lisa",
    "apple",
    "banana",
    "cat",
    "dog",
    "fish",
    "bird",
    "airplane",
    "car",
    "bicycle",
    "bus",
    "tree",
    "flower",
    "house",
    "chair",
    "table",
    "cup",
    "fork",
    "umbrella",
    "star",
    "moon",
    "sun",
    "cloud",
    "crown",
    "pizza",
    "ice cream",
    "book",
    "clock",
    "eye",
    "face"
  ],

In [16]:
!cd /content && zip -r quickdraw_fused_tflite_export.zip quickdraw_fused_tflite

from google.colab import files
files.download("/content/quickdraw_fused_tflite_export.zip")

  adding: quickdraw_fused_tflite/ (stored 0%)
  adding: quickdraw_fused_tflite/history.json (deflated 65%)
  adding: quickdraw_fused_tflite/best.keras (deflated 10%)
  adding: quickdraw_fused_tflite/quickdraw_fused_model_float32.tflite (deflated 8%)
  adding: quickdraw_fused_tflite/final.keras (deflated 10%)
  adding: quickdraw_fused_tflite/labels.json (deflated 53%)
  adding: quickdraw_fused_tflite/model_info.json (deflated 61%)
  adding: quickdraw_fused_tflite/quickdraw_fused_model_float16.tflite (deflated 10%)


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>